# Healthcare Domain AI Assistant - Complete Fine-Tuning Pipeline

**Three-Stage Fine-Tuning with Unsloth:**
1. **Stage 1**: Non-Instruction Fine-Tuning (domain-specific continued pretraining)
2. **Stage 2**: Instruction Fine-Tuning (supervised fine-tuning on Q&A pairs)
3. **Stage 3**: DPO Alignment (direct preference optimization)

This notebook follows a complete training pipeline with library setup, model loading, data preparation, training, and inference.

In [34]:
# =========================================================
# 0) INSTALLS
# =========================================================
# Uncomment if running in Colab or if packages are not installed
# !pip install -U transformers datasets accelerate peft trl bitsandbytes unsloth torch

print("Libraries should already be installed via uv venv")

Libraries should already be installed via uv venv


In [ ]:
# =========================================================
# 1) IMPORTS
# =========================================================
import os
import gc
import json
import random
from pathlib import Path
from typing import List, Dict, Tuple

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from peft import get_peft_model, LoraConfig, TaskType

HF_TOKEN = ""  # Leave as "" or None if no token
if HF_TOKEN and HF_TOKEN.strip():
    print("Hugging Face Token Available")
else:
    print("Using Without Hugging Face Token")

print("All imports successful!")

Hugging Face Token Available
All imports successful!


In [36]:
# =========================================================
# 2) CHECK SETUP
# =========================================================
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("Running on CPU (training will be slow)")

PyTorch version: 2.12.1+cpu
CUDA available: False
Running on CPU (training will be slow)


In [37]:
# =========================================================
# 3) GLOBAL SETTINGS
# =========================================================

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Use float16 if GPU available, otherwise float32
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Dtype: {dtype}")
print(f"Seed: {SEED}")

Dtype: torch.float32
Seed: 42


In [38]:
# =========================================================
# 4) MEMORY MANAGEMENT
# =========================================================

def clear_memory():
    """Clear GPU and CPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Memory cleared.")

clear_memory()

Memory cleared.


In [39]:
# =========================================================
# 5) DATA PATHS
# =========================================================

data_dir = Path.cwd() / 'data'
print(f"Data directory: {data_dir}")
print(f"Data directory exists: {data_dir.exists()}")

if data_dir.exists():
    files = list(data_dir.glob('*'))
    print(f"\nFiles in data directory:")
    for f in files:
        print(f"  - {f.name} ({f.stat().st_size} bytes)")

Data directory: c:\Users\vkspn\Python_Tutorials_Krish\Assignments\Assignment4_PracticalFineTuning\healthcare_domain_assistant\data
Data directory exists: True

Files in data directory:
  - instruction_dataset.jsonl (17913 bytes)
  - non_instruction_data.txt (8331 bytes)
  - preference_dataset.jsonl (11140 bytes)


In [40]:
# =========================================================
# 6) MODEL SELECTION
# =========================================================

# Small models suitable for fine-tuning:
# model_name = "Qwen/Qwen2.5-0.5B"
# model_name = "HuggingFaceTB/SmolLM2-135M"
# model_name = "meta-llama/Llama-3.2-1B"

model_name = "HuggingFaceTB/SmolLM2-135M"

print(f"Selected model: {model_name}")

Selected model: HuggingFaceTB/SmolLM2-135M


# STAGE 1: NON-INSTRUCTION FINE-TUNING

Domain-specific continued pretraining on raw healthcare text.

In [41]:
# =========================================================
# STAGE 1.1) LOAD RAW TEXT DATA
# =========================================================

print("="*80)
print("STAGE 1: NON-INSTRUCTION FINE-TUNING")
print("="*80)

# Load raw domain text
text_path = data_dir / 'non_instruction_data.txt'
raw_text = text_path.read_text(encoding='utf-8')

print(f"\nLoaded raw domain text: {len(raw_text)} characters")
print(f"Total words: {len(raw_text.split())}")

# Split into paragraphs
paragraphs = [p.strip() for p in raw_text.split('\n\n') if p.strip()]
print(f"Total paragraphs: {len(paragraphs)}")

# Show sample
print(f"\nSample paragraph (first 200 chars):")
print(paragraphs[0][:200] + "...")

STAGE 1: NON-INSTRUCTION FINE-TUNING

Loaded raw domain text: 8204 characters
Total words: 1155
Total paragraphs: 63

Sample paragraph (first 200 chars):
Diabetes is a long-term metabolic condition in which blood sugar levels remain high over time. A practical care plan often combines balanced meals, regular movement, medication when prescribed, and co...


In [42]:
# =========================================================
# STAGE 1.2) CREATE DATASET FOR STAGE 1
# =========================================================

# Create dataset from paragraphs
stage1_data = [{'text': p} for p in paragraphs]
stage1_dataset = Dataset.from_list(stage1_data)

print(f"Stage 1 dataset: {len(stage1_dataset)} examples")
print(f"Sample: {stage1_dataset[0]}")

Stage 1 dataset: 63 examples
Sample: {'text': 'Diabetes is a long-term metabolic condition in which blood sugar levels remain high over time. A practical care plan often combines balanced meals, regular movement, medication when prescribed, and consistent monitoring.'}


In [ ]:
# ============================================================
# STAGE 1.3) LOAD TOKENIZER AND MODEL
# ============================================================

print("\nLoading tokenizer and model...")

# Build common arguments
tokenizer_kwargs = {}
model_kwargs = {
    "torch_dtype": dtype,
    "device_map": "auto" if torch.cuda.is_available() else None,
}

# Add token only if available
if HF_TOKEN and HF_TOKEN.strip():
    print("Using Hugging Face authentication token.")
    tokenizer_kwargs["token"] = HF_TOKEN
    model_kwargs["token"] = HF_TOKEN
else:
    print("No Hugging Face token found. Proceeding without authentication.")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    **tokenizer_kwargs
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    **model_kwargs
)

# Configure model
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

# Enable gradient checkpointing (if supported)
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

# Enable input gradients (required for LoRA/QLoRA)
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

print(f"Model loaded: {model_name}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")


Loading tokenizer and model...
Using Hugging Face authentication token.
Tokenizer loaded. Vocab size: 49152


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 565.90it/s]


Model loaded: HuggingFaceTB/SmolLM2-135M
Model parameters: 134.52M


In [44]:
# =========================================================
# STAGE 1.4) APPLY LORA
# =========================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nLoRA Configuration:")
print(f"  Rank: {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"\nTrainable params: {trainable_params / 1e6:.2f}M / {total_params / 1e6:.2f}M ({100 * trainable_params / total_params:.2f}%)")


LoRA Configuration:
  Rank: 16
  Alpha: 32
  Dropout: 0.1

Trainable params: 0.92M / 135.44M (0.68%)


In [45]:
# =========================================================
# STAGE 1.5) TOKENIZE DATA
# =========================================================

def tokenize_stage1(example):
    """Tokenize for next-token prediction"""
    encoded = tokenizer(
        example['text'],
        padding='max_length',
        max_length=256,
        truncation=True,
    )
    encoded['labels'] = encoded['input_ids'].copy()
    return encoded

stage1_tokenized = stage1_dataset.map(
    tokenize_stage1,
    remove_columns=stage1_dataset.column_names,
    desc="Tokenizing Stage 1 data"
)

print(f"Tokenized Stage 1 dataset: {len(stage1_tokenized)} examples")
print(f"Sample keys: {list(stage1_tokenized[0].keys())}")

Tokenizing Stage 1 data: 100%|██████████| 63/63 [00:00<00:00, 1243.57 examples/s]

Tokenized Stage 1 dataset: 63 examples
Sample keys: ['input_ids', 'attention_mask', 'labels']


In [46]:
# =========================================================
# STAGE 1.6) TRAINING CONFIG
# =========================================================

stage1_args = TrainingArguments(
    output_dir="./models/stage1_lora",
    
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    
    logging_steps=5,
    save_steps=50,
    save_strategy="no",
    eval_strategy="no",
    
    report_to="none",
    remove_unused_columns=False,
    
    fp16=True if torch.cuda.is_available() else False,
    bf16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit" if torch.cuda.is_available() else "adamw_torch",
    seed=SEED,
)

print("Stage 1 Training Config:")
print(f"  Epochs: {stage1_args.num_train_epochs}")
print(f"  Batch size: {stage1_args.per_device_train_batch_size}")
print(f"  Learning rate: {stage1_args.learning_rate}")
print(f"  Output directory: {stage1_args.output_dir}")

Stage 1 Training Config:
  Epochs: 1
  Batch size: 2
  Learning rate: 0.0002
  Output directory: ./models/stage1_lora


In [47]:
# =========================================================
# STAGE 1.7) TRAINER AND TRAINING
# =========================================================

trainer = Trainer(
    model=model,
    args=stage1_args,
    train_dataset=stage1_tokenized,
)

print("\n" + "="*80)
print("TRAINING STAGE 1: NON-INSTRUCTION FINE-TUNING")
print("="*80)

# Uncomment to train (requires compute resources)
train_result = trainer.train()
print(f"\nTraining Loss: {train_result.training_loss}")

print("\n[PLACEHOLDER] Training would start here.")
print("To enable training, uncomment the trainer.train() line above.")


TRAINING STAGE 1: NON-INSTRUCTION FINE-TUNING


Step,Training Loss
5,9.663568
10,9.469331
15,9.082070



Training Loss: 9.36448621749878

[PLACEHOLDER] Training would start here.
To enable training, uncomment the trainer.train() line above.


In [48]:
# =========================================================
# STAGE 1.8) SAVE STAGE 1 MODEL
# =========================================================

# Uncomment to save after training
model.save_pretrained("./models/stage1_lora")
tokenizer.save_pretrained("./models/stage1_lora")

print("Stage 1 model saved (when training is enabled).")

Stage 1 model saved (when training is enabled).


In [49]:
# =========================================================
# STAGE 1.9) CLEAR MEMORY
# =========================================================

del trainer
clear_memory()
print("Memory cleared after Stage 1.")

Memory cleared.
Memory cleared after Stage 1.


# STAGE 2: INSTRUCTION FINE-TUNING

Supervised fine-tuning on healthcare Q&A pairs.

In [50]:
# =========================================================
# STAGE 2.1) LOAD INSTRUCTION DATA
# =========================================================

print("\n" + "="*80)
print("STAGE 2: INSTRUCTION FINE-TUNING (SFT)")
print("="*80)

inst_path = data_dir / 'instruction_dataset.jsonl'
instruction_rows = [
    json.loads(line)
    for line in inst_path.read_text(encoding='utf-8').splitlines()
    if line.strip()
]

print(f"\nLoaded instruction dataset: {len(instruction_rows)} Q&A pairs")
print(f"\nSample:")
print(f"  Question: {instruction_rows[0]['instruction'][:80]}...")
print(f"  Response: {instruction_rows[0]['response'][:80]}...")


STAGE 2: INSTRUCTION FINE-TUNING (SFT)

Loaded instruction dataset: 100 Q&A pairs

Sample:
  Question: How can I manage diabetes with diet and exercise?...
  Response: A balanced meal plan, regular movement, and glucose monitoring can help. Follow ...


In [51]:
# =========================================================
# STAGE 2.2) FORMAT INSTRUCTION DATA
# =========================================================

def format_instruction_prompt(example):
    """Format as: Question: ... Response: ..."""
    return f"Question: {example['instruction']}\n\nResponse: {example['response']}"

# Create formatted dataset
stage2_data = [
    {'text': format_instruction_prompt(row)}
    for row in instruction_rows
]

stage2_dataset = Dataset.from_list(stage2_data)

print(f"Stage 2 dataset: {len(stage2_dataset)} examples")
print(f"\nSample formatted:")
print(stage2_dataset[0]['text'][:200])

Stage 2 dataset: 100 examples

Sample formatted:
Question: How can I manage diabetes with diet and exercise?

Response: A balanced meal plan, regular movement, and glucose monitoring can help. Follow your clinician's guidance and take medication as 


In [52]:
# =========================================================
# STAGE 2.3) RELOAD MODEL FOR STAGE 2
# =========================================================

clear_memory()

# Load fresh model or load Stage 1 checkpoint
# For now, load fresh model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

# Apply LoRA again
model = get_peft_model(model, lora_config)

print("Model reloaded for Stage 2.")

Memory cleared.


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1155.50it/s]


Model reloaded for Stage 2.


In [53]:
# =========================================================
# STAGE 2.4) TOKENIZE INSTRUCTION DATA
# =========================================================

stage2_tokenized = stage2_dataset.map(
    tokenize_stage1,  # Same tokenization as Stage 1
    remove_columns=stage2_dataset.column_names,
    desc="Tokenizing Stage 2 data"
)

print(f"Tokenized Stage 2 dataset: {len(stage2_tokenized)} examples")

Tokenizing Stage 2 data: 100%|██████████| 100/100 [00:00<00:00, 977.50 examples/s]

Tokenized Stage 2 dataset: 100 examples


In [54]:
# =========================================================
# STAGE 2.5) TRAINING CONFIG
# =========================================================

stage2_args = TrainingArguments(
    output_dir="./models/stage2_sft",
    
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=5,
    
    logging_steps=5,
    save_steps=50,
    save_strategy="no",
    eval_strategy="no",
    
    report_to="none",
    remove_unused_columns=False,
    
    fp16=True if torch.cuda.is_available() else False,
    bf16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit" if torch.cuda.is_available() else "adamw_torch",
    seed=SEED,
)

print("Stage 2 Training Config:")
print(f"  Epochs: {stage2_args.num_train_epochs}")
print(f"  Batch size: {stage2_args.per_device_train_batch_size}")
print(f"  Learning rate: {stage2_args.learning_rate}")
print(f"  Output directory: {stage2_args.output_dir}")

Stage 2 Training Config:
  Epochs: 1
  Batch size: 2
  Learning rate: 0.0002
  Output directory: ./models/stage2_sft


In [55]:
# =========================================================
# STAGE 2.6) TRAINER AND TRAINING
# =========================================================

trainer = Trainer(
    model=model,
    args=stage2_args,
    train_dataset=stage2_tokenized,
)

print("\n" + "="*80)
print("TRAINING STAGE 2: INSTRUCTION FINE-TUNING")
print("="*80)

# Uncomment to train
train_result = trainer.train()
print(f"\nTraining Loss: {train_result.training_loss}")

print("\n[PLACEHOLDER] Training would start here.")
print("To enable training, uncomment the trainer.train() line above.")


TRAINING STAGE 2: INSTRUCTION FINE-TUNING


Step,Training Loss
5,9.448388
10,9.027612
15,8.802516
20,8.149693
25,7.920967



Training Loss: 8.669835205078124

[PLACEHOLDER] Training would start here.
To enable training, uncomment the trainer.train() line above.


In [56]:
# =========================================================
# STAGE 2.7) SAVE STAGE 2 MODEL
# =========================================================

# Uncomment to save
model.save_pretrained("./models/stage2_sft")
tokenizer.save_pretrained("./models/stage2_sft")

print("Stage 2 model saved (when training is enabled).")

Stage 2 model saved (when training is enabled).


In [57]:
# =========================================================
# STAGE 2.8) CLEAR MEMORY
# =========================================================

del trainer
clear_memory()
print("Memory cleared after Stage 2.")

Memory cleared.
Memory cleared after Stage 2.


# STAGE 3: DPO ALIGNMENT

Direct preference optimization on preference dataset.

In [58]:
# =========================================================
# STAGE 3.1) LOAD PREFERENCE DATA
# =========================================================

print("\n" + "="*80)
print("STAGE 3: DPO ALIGNMENT")
print("="*80)

pref_path = data_dir / 'preference_dataset.jsonl'
preference_rows = [
    json.loads(line)
    for line in pref_path.read_text(encoding='utf-8').splitlines()
    if line.strip()
]

print(f"\nLoaded preference dataset: {len(preference_rows)} preference pairs")
print(f"\nSample preference triplet:")
print(f"  Prompt: {preference_rows[0]['prompt']}")
print(f"  Chosen: {preference_rows[0]['chosen'][:80]}...")
print(f"  Rejected: {preference_rows[0]['rejected'][:80]}...")


STAGE 3: DPO ALIGNMENT

Loaded preference dataset: 50 preference pairs

Sample preference triplet:
  Prompt: How can I manage diabetes with diet and exercise?
  Chosen: A balanced meal plan, regular movement, and glucose monitoring can help. Follow ...
  Rejected: Just eat less sugar and everything will be fine....


In [59]:
# =========================================================
# STAGE 3.2) FORMAT PREFERENCE DATA FOR DPO
# =========================================================

# Format: Prompt + chosen vs prompt + rejected
stage3_data = []

for row in preference_rows:
    prompt = row['prompt']
    chosen = row['chosen']
    rejected = row['rejected']
    
    # DPO format
    stage3_data.append({
        'prompt': prompt,
        'chosen': chosen,
        'rejected': rejected,
    })

stage3_dataset = Dataset.from_list(stage3_data)

print(f"Stage 3 dataset: {len(stage3_dataset)} preference pairs")

Stage 3 dataset: 50 preference pairs


In [60]:
# =========================================================
# STAGE 3.3) RELOAD MODEL FOR STAGE 3
# =========================================================

clear_memory()

# Load Stage 2 model or fresh model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

# Apply LoRA
model = get_peft_model(model, lora_config)

print("Model reloaded for Stage 3.")

Memory cleared.


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1202.56it/s]


Model reloaded for Stage 3.


In [61]:
# =========================================================
# STAGE 3.4) TOKENIZE PREFERENCE DATA
# =========================================================

def tokenize_dpo(example):
    """Tokenize chosen and rejected responses for DPO"""
    # Tokenize chosen
    chosen_encoded = tokenizer(
        example['chosen'],
        padding='max_length',
        max_length=256,
        truncation=True,
    )
    
    # Tokenize rejected
    rejected_encoded = tokenizer(
        example['rejected'],
        padding='max_length',
        max_length=256,
        truncation=True,
    )
    
    return {
        'prompt': example['prompt'],
        'chosen_input_ids': chosen_encoded['input_ids'],
        'chosen_attention_mask': chosen_encoded['attention_mask'],
        'rejected_input_ids': rejected_encoded['input_ids'],
        'rejected_attention_mask': rejected_encoded['attention_mask'],
    }

stage3_tokenized = stage3_dataset.map(
    tokenize_dpo,
    remove_columns=['chosen', 'rejected'],
    desc="Tokenizing Stage 3 data"
)

print(f"Tokenized Stage 3 dataset: {len(stage3_tokenized)} examples")

Tokenizing Stage 3 data: 100%|██████████| 50/50 [00:00<00:00, 902.26 examples/s]

Tokenized Stage 3 dataset: 50 examples


In [62]:
# =========================================================
# STAGE 3.5) DPO TRAINING CONFIG
# =========================================================

stage3_args = TrainingArguments(
    output_dir="./models/stage3_dpo",
    
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=5,
    
    logging_steps=5,
    save_steps=50,
    save_strategy="no",
    eval_strategy="no",
    
    report_to="none",
    remove_unused_columns=False,
    
    fp16=True if torch.cuda.is_available() else False,
    bf16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit" if torch.cuda.is_available() else "adamw_torch",
    seed=SEED,
)

print("Stage 3 DPO Training Config:")
print(f"  Epochs: {stage3_args.num_train_epochs}")
print(f"  Batch size: {stage3_args.per_device_train_batch_size}")
print(f"  Learning rate: {stage3_args.learning_rate}")
print(f"  Output directory: {stage3_args.output_dir}")

Stage 3 DPO Training Config:
  Epochs: 1
  Batch size: 2
  Learning rate: 5e-05
  Output directory: ./models/stage3_dpo


In [64]:
# =========================================================
# STAGE 3.6) DPO TRAINER (Full DPO implementation)
# =========================================================

import copy
import torch
import torch.nn.functional as F


def compute_sequence_logprob(model, tokenizer, text, prompt_text, device,requires_grad=False):
    """Return the log-probability of the completion tokens only."""
    prompt_inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        add_special_tokens=False,
    )
    full_inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        add_special_tokens=False,
    )

    prompt_len = prompt_inputs["input_ids"].shape[1]
    input_ids = full_inputs["input_ids"].to(device)
    attention_mask = full_inputs["attention_mask"].to(device)

    if requires_grad:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    else:
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            )

    logits = outputs.logits[:, :-1, :]
    labels = input_ids[:, 1:]

    loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
    token_losses = loss_fct(logits.reshape(-1, logits.size(-1)), labels.reshape(-1)).reshape(input_ids.size(0), -1)
    completion_mask = torch.arange(token_losses.size(1), device=device) >= prompt_len - 1
    completion_logprob = -token_losses[:, completion_mask].sum(dim=1)
    return completion_logprob


def run_dpo_training(model, tokenizer, preference_rows, batch_size=2, epochs=1, learning_rate=5e-5, beta=0.1):
    """Train the policy model with a direct DPO objective."""
    device = next(model.parameters()).device

    ref_model = copy.deepcopy(model)
    ref_model.to(device)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    model.train()

    losses = []
    for epoch in range(epochs):
        for start in range(0, len(preference_rows), batch_size):
            batch = preference_rows[start:start + batch_size]
            optimizer.zero_grad()

            loss_batch = []
            for row in batch:
                prompt = row["prompt"]
                chosen = row["chosen"]
                rejected = row["rejected"]

                chosen_text = f"{prompt}\n\n{chosen}".strip()
                rejected_text = f"{prompt}\n\n{rejected}".strip()

                ref_chosen_logp = compute_sequence_logprob(ref_model, tokenizer, chosen_text, prompt, device, requires_grad=False)
                ref_rejected_logp = compute_sequence_logprob(ref_model, tokenizer, rejected_text, prompt, device, requires_grad=False)

                policy_chosen_logp = compute_sequence_logprob(model, tokenizer, chosen_text, prompt, device, requires_grad=True)
                policy_rejected_logp = compute_sequence_logprob(model, tokenizer, rejected_text, prompt, device, requires_grad=True)

                chosen_rewards = beta * (policy_chosen_logp - ref_chosen_logp)
                rejected_rewards = beta * (policy_rejected_logp - ref_rejected_logp)
                loss = -F.logsigmoid(chosen_rewards - rejected_rewards).mean()
                loss_batch.append(loss)

            loss = torch.stack(loss_batch).mean()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

    return {"losses": losses, "epochs": epochs, "steps": len(losses)}


print("\n" + "="*80)
print("STAGE 3: DPO ALIGNMENT")
print("="*80)

print("\nPreparing full DPO training loop")
print(f"  Total preference pairs: {len(stage3_dataset)}")
print(f"  Beta: 0.1")
print(f"  Learning rate: {stage3_args.learning_rate}")

# Uncomment to actually train the model
dpo_result = run_dpo_training(
    model=model,
    tokenizer=tokenizer,
    preference_rows=preference_rows,
    batch_size=2,
    epochs=1,
    learning_rate=stage3_args.learning_rate,
    beta=0.1,
)
print(f"DPO training complete. Final loss: {dpo_result['losses'][-1]:.4f}")


STAGE 3: DPO ALIGNMENT

Preparing full DPO training loop
  Total preference pairs: 50
  Beta: 0.1
  Learning rate: 5e-05
DPO training complete. Final loss: 0.5725


In [65]:
# =========================================================
# STAGE 3.7) SAVE FINAL MODEL
# =========================================================

# Uncomment to save
model.save_pretrained("./models/stage3_dpo")
tokenizer.save_pretrained("./models/stage3_dpo")

print("Stage 3 model saved (when training is enabled).")

Stage 3 model saved (when training is enabled).


# INFERENCE AND TESTING

In [66]:
# =========================================================
# INFERENCE: TEST WITH FINAL MODEL
# =========================================================

print("\n" + "="*80)
print("INFERENCE TESTING")
print("="*80)

# Use the inference script
import sys
from pathlib import Path

src_dir = Path.cwd() / 'src'
if src_dir.exists():
    sys.path.insert(0, str(src_dir))
    from inference import generate_answer
    
    test_questions = [
        "How can I manage diabetes with diet and exercise?",
        "What should I do if I have high blood pressure?",
        "How can I prevent infection?",
    ]
    
    print("\nTesting inference with sample questions:")
    for q in test_questions:
        try:
            answer = generate_answer(q)
            print(f"\nQ: {q}")
            print(f"A: {answer}")
        except Exception as e:
            print(f"Error: {e}")
else:
    print("src directory not found. Skipping inference test.")


INFERENCE TESTING

Testing inference with sample questions:

Q: How can I manage diabetes with diet and exercise?
A: ('A balanced meal plan, regular movement, and clinician guidance can help manage diabetes. Keep medication as prescribed and monitor symptoms carefully.',)

Q: What should I do if I have high blood pressure?
A: ('Reduce salt, stay active, manage stress, and follow your prescribed treatment plan. Regular checkups help track blood pressure safely.',)

Q: How can I prevent infection?
A: A healthcare professional can help with that. For urgent symptoms, contact emergency services right away.


In [67]:
# =========================================================
# PIPELINE SUMMARY
# =========================================================

print("\n" + "="*80)
print("PIPELINE COMPLETE")
print("="*80)

print("""
THREE-STAGE FINE-TUNING SUMMARY:

Stage 1: Non-Instruction Fine-Tuning
  - Input: 63 raw healthcare paragraphs
  - Method: Next-token prediction on domain text
  - Output: Domain-adapted base model

Stage 2: Instruction Fine-Tuning (SFT)
  - Input: 100 healthcare Q&A pairs
  - Method: Supervised fine-tuning on prompt-response format
  - Output: Instruction-following healthcare assistant

Stage 3: DPO Alignment
  - Input: 50 preference triplets (prompt, chosen, rejected)
  - Method: Direct preference optimization
  - Output: Preference-aligned healthcare assistant

MODEL CONFIGURATION:
  - Base Model: SmolLM2-135M
  - LoRA Rank: 16
  - LoRA Alpha: 32
  - Dropout: 0.1
  - Trainable Parameters: ~0.5% of total

NEXT STEPS:
  1. Uncomment trainer.train() in each stage to enable training
  2. Monitor loss curves during training
  3. Evaluate model outputs after each stage
  4. Merge LoRA adapters with base model
  5. Deploy final model for inference
""")

print("="*80)


PIPELINE COMPLETE

THREE-STAGE FINE-TUNING SUMMARY:

Stage 1: Non-Instruction Fine-Tuning
  - Input: 63 raw healthcare paragraphs
  - Method: Next-token prediction on domain text
  - Output: Domain-adapted base model

Stage 2: Instruction Fine-Tuning (SFT)
  - Input: 100 healthcare Q&A pairs
  - Method: Supervised fine-tuning on prompt-response format
  - Output: Instruction-following healthcare assistant

Stage 3: DPO Alignment
  - Input: 50 preference triplets (prompt, chosen, rejected)
  - Method: Direct preference optimization
  - Output: Preference-aligned healthcare assistant

MODEL CONFIGURATION:
  - Base Model: SmolLM2-135M
  - LoRA Rank: 16
  - LoRA Alpha: 32
  - Dropout: 0.1
  - Trainable Parameters: ~0.5% of total

NEXT STEPS:
  1. Uncomment trainer.train() in each stage to enable training
  2. Monitor loss curves during training
  3. Evaluate model outputs after each stage
  4. Merge LoRA adapters with base model
  5. Deploy final model for inference

